# PointNet

In [1]:
from typing import Tuple, Optional
import torch
import torch.nn as nn
from torch import Tensor
from torch_scatter import scatter_max, scatter_mean

## Classification

In [ ]:
class TNet(nn.Module):
    def __init__(self, k: int) -> None:
        """Input Transform Net or Feature Transform Net
        
        Args:
            k: Input dimension (3 for input transform, n_features for feature transform)
        """
        super().__init__()
        self.k = k
        
        self.mlp = nn.Sequential(
            nn.Linear(k, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU()
        )
        
        self.regressor = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, k * k)
        )
        
    def forward(self, x: Tensor, batch: Tensor) -> Tensor:
        """
        Args:
            x: Point cloud features (N, k)
            batch: Batch assignments (N,)
            
        Returns:
            transform_matrix: Transformation matrix (B, k, k)
        """
        x = self.mlp(x)
        x = scatter_max(x, batch, dim=0)[0]
        
        x = self.regressor(x)
        transform_matrix = x.view(-1, self.k, self.k)
        
        # Add identity to make it easier to learn identity-like transformations
        identity = torch.eye(self.k, dtype=x.dtype, device=x.device)
        transform_matrix = transform_matrix + identity.unsqueeze(0)
        
        return transform_matrix

    
    
class PointNet(nn.Module):
    def __init__(
        self, 
        in_channels: int = 3,
        feature_channels: int = 64,
        num_classes: int = 40,
        use_feature_transform: bool = True
    ) -> None:
        """PointNet for Classification
        
        Args:
            in_channels: Number of input channels (default: 3 for XYZ)
            feature_channels: Number of channels in point features
            num_classes: Number of output classes
            use_feature_transform: Whether to use feature transform
        """
        super().__init__()
        
        self.input_transform = TNet(k=in_channels)
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_channels, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, feature_channels),
            nn.BatchNorm1d(feature_channels),
            nn.ReLU()
        )
        
        self.use_feature_transform = use_feature_transform
        if use_feature_transform:
            self.feature_transform = TNet(k=feature_channels)
            
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_channels, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU()
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(
        self, 
        points: Tensor,
        batch: Tensor,
        features: Optional[Tensor] = None
    ) -> Tuple[Tensor, Optional[Tensor]]:
        """
        Args:
            points: Point cloud coordinates (N, in_channels)
            batch: Batch assignments (N,)
            features: Optional point features (N, C)
            
        Returns:
            logits: Classification logits (B, num_classes)
            feature_transform: Feature transform matrix if use_feature_transform=True
        """
        # Input transform on coordinates
        input_transform = self.input_transform(points, batch)
        xs = torch.bmm(
            points.view(-1, 1, self.input_transform.k),
            input_transform[batch]
        ).view(-1, self.input_transform.k)
        
        # Feature extraction
        x = torch.cat([xs, features], dim=1) if features is not None else xs
        print(f"{x.shape=}")
        x = self.feature_extractor(x)
        
        # if features is not None:
        #     # If features are provided, concatenate transformed coordinates with features
        #     x = torch.cat([xs, features], dim=1)
        #     x = self.feature_extractor(x)
        # else:
        #     # Use only transformed coordinates
        #     x = self.feature_extractor(xs)
        
        # Feature transform if enabled
        feature_transform = None
        if self.use_feature_transform:
            feature_transform = self.feature_transform(x, batch)
            x = torch.bmm(
                x.view(-1, 1, feature_transform.size(2)),
                feature_transform[batch]
            ).view(-1, feature_transform.size(1))
        
        # Global feature learning
        x = self.feature_mlp(x)
        x = scatter_max(x, batch, dim=0)[0]  # (B, 1024)
        
        # Classification
        logits = self.classifier(x)
        
        return logits, feature_transform

In [34]:
def pointnet_regularization_loss(feature_transform: Tensor) -> Tensor:
    """Compute regularization loss for the feature transform matrix
    
    Args:
        feature_transform: Feature transform matrix (B, K, K)
        
    Returns:
        reg_loss: Regularization loss
    """
    B, K, _ = feature_transform.size()
    I = torch.eye(K, dtype=feature_transform.dtype, device=feature_transform.device)
    I = I.unsqueeze(0).expand(B, K, K)
    
    # Compute A*A^T - I
    matmul = torch.bmm(feature_transform, feature_transform.transpose(2, 1))
    diff = matmul - I
    
    # Compute Frobenius norm
    reg_loss = torch.mean(torch.norm(diff, dim=(1, 2)))
    return reg_loss

In [35]:
# In packed mode
points = torch.randn(100, 3)
features = torch.randn(100, 64)
batch = torch.tensor([0] * 60 + [1] * 40)


model = PointNet(in_channels=3, feature_channels=64, num_classes=40, use_feature_transform=True)
logits, feature_transform = model(points, batch, None)

x.shape=torch.Size([100, 3])


In [37]:
model.feature_extractor(points)

tensor([[0.0000, 0.0000, 0.8997,  ..., 0.9892, 0.5756, 0.4504],
        [0.0000, 0.0000, 0.8993,  ..., 1.2972, 0.0000, 0.6477],
        [0.0000, 0.1385, 0.0778,  ..., 0.2524, 0.6147, 1.0593],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.8540, 1.3647, 0.0000],
        [0.0000, 0.0000, 0.5850,  ..., 0.0000, 0.1525, 0.6691],
        [0.0000, 0.0000, 0.7136,  ..., 1.5293, 0.0000, 0.0000]],
       grad_fn=<ReluBackward0>)